# 08 - Phase 2 Data Collection

Run the Phase 2 experiment matrix: DQN, DDQN, and PPO across sparse, subgoal, and potential rewards. The notebook starts in smoke-test mode. Change `SMOKE_TEST = False` for the full experiment.

In [27]:
from pathlib import Path
import importlib
import json
import sys
import time

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np

import src.env.door_key_maze_env as door_key_maze_env
import src.utils.metrics_phase2 as metrics_phase2
import src.training.trainer_phase2 as trainer_phase2
door_key_maze_env = importlib.reload(door_key_maze_env)
metrics_phase2 = importlib.reload(metrics_phase2)
trainer_phase2 = importlib.reload(trainer_phase2)
from src.env import DoorKeyMazeEnv
DoorKeyMazeEnv = door_key_maze_env.DoorKeyMazeEnv
DQN_CFG = trainer_phase2.DQN_CFG
PPO_CFG = trainer_phase2.PPO_CFG
train_dqn_phase2 = trainer_phase2.train_dqn_phase2
train_ppo_phase2 = trainer_phase2.train_ppo_phase2
from src.utils.io import DIRS, ensure_dirs, save_metrics, save_dqn

ensure_dirs()

## Single Source Of Truth

In [28]:
SMOKE_TEST = True
RUN_LABEL = "smoke"

MAZE_KWARGS = dict(level="small", difficulty="medium", seed=42, dynamic_objects=True, n_dynamic_layouts=4)
ALGORITHMS = ["dqn", "ddqn", "ppo"]
REWARDS = ["sparse", "subgoal", "potential"]
SEEDS = [0, 1, 2, 3, 4]

if SMOKE_TEST:
    ALGORITHMS_TO_RUN = ["ddqn"]
    REWARDS_TO_RUN = ["subgoal"]
    SEEDS_TO_RUN = [0]
    DQN_RUN_CFG = {**DQN_CFG, "episodes": 10, "warmup": 10}
    PPO_RUN_CFG = {**PPO_CFG, "total_timesteps": 1024, "n_steps": 256}
else:
    ALGORITHMS_TO_RUN = ALGORITHMS
    REWARDS_TO_RUN = REWARDS
    SEEDS_TO_RUN = SEEDS
    DQN_RUN_CFG = DQN_CFG
    PPO_RUN_CFG = PPO_CFG

print("algorithms:", ALGORITHMS_TO_RUN)
print("rewards:", REWARDS_TO_RUN)
print("seeds:", SEEDS_TO_RUN)

algorithms: ['ddqn']
rewards: ['subgoal']
seeds: [0]


## Save Helpers

In [29]:
def run_id_for(algo, reward, seed):
    mode = globals().get("RUN_LABEL", "smoke" if SMOKE_TEST else "full")
    return f"p2_{mode}_{algo}_{reward}_{MAZE_KWARGS['level']}_{MAZE_KWARGS['difficulty']}_seed{seed}"

def tracker_optimal_path_arr(tracker):
    if hasattr(tracker, "optimal_path_arr"):
        return tracker.optimal_path_arr
    return np.full_like(tracker.steps_arr, tracker.optimal_path_len, dtype=float)

def save_phase2_result(run_id, env, result, algo, reward, seed):
    tracker = result.tracker
    metrics_path = save_metrics({
        "episodes": tracker.episodes_arr,
        "rewards": tracker.rewards,
        "steps": tracker.steps_arr,
        "success": tracker.success_arr,
        "key": tracker.key_arr,
        "door": tracker.door_arr,
        "steps_to_key": tracker.steps_to_key_arr,
        "steps_to_door": tracker.steps_to_door_arr,
        "optimal_path": tracker_optimal_path_arr(tracker),
    }, run_id)
    if algo in {"dqn", "ddqn"}:
        save_dqn(result.model, run_id)
    summary = tracker.compute_summary()
    summary.update({
        "run_id": run_id,
        "algo": algo,
        "reward": reward,
        "seed": seed,
        "maze_level": env.level,
        "maze_size": env.size,
        "difficulty": env.difficulty,
        "key_pos": list(env.key_pos),
        "door_pos": list(env.door_pos),
        "dynamic_objects": env.dynamic_objects,
        "n_dynamic_layouts": len(env._layouts),
        "metrics_path": str(metrics_path),
    })
    summary_path = DIRS["logs"] / f"{run_id}_summary.json"
    summary_path.write_text(json.dumps(summary, indent=2), encoding="utf-8")
    return summary

def evaluate_ppo_policy(env, model, episodes=100):
    rewards, steps, success, key, door, steps_to_key, steps_to_door, optimal_path = [], [], [], [], [], [], [], []
    for _ in range(episodes):
        obs, _ = env.reset()
        total = 0.0
        done = False
        info = {}
        while not done:
            action, _ = model.predict(obs, deterministic=True)
            obs, reward, terminated, truncated, info = env.step(int(action))
            total += float(reward)
            done = terminated or truncated
        rewards.append(total)
        steps.append(env.steps_taken)
        success.append(int(terminated))
        key.append(info.get("picked_up_key", 0))
        door.append(info.get("opened_door", 0))
        steps_to_key.append(np.nan if info.get("steps_to_key") is None else info.get("steps_to_key"))
        steps_to_door.append(np.nan if info.get("steps_to_door") is None else info.get("steps_to_door"))
        optimal_path.append(info.get("optimal_path_len", np.nan))
    return {
        "episodes": np.arange(1, episodes + 1),
        "rewards": np.array(rewards, dtype=float),
        "steps": np.array(steps, dtype=float),
        "success": np.array(success, dtype=float),
        "key": np.array(key, dtype=float),
        "door": np.array(door, dtype=float),
        "steps_to_key": np.array(steps_to_key, dtype=float),
        "steps_to_door": np.array(steps_to_door, dtype=float),
        "optimal_path": np.array(optimal_path, dtype=float),
    }

def save_ppo_summary(run_id, env, model, key_log, door_log, algo, reward, seed):
    eval_episodes = 5 if SMOKE_TEST else 100
    eval_metrics = evaluate_ppo_policy(env, model, episodes=eval_episodes)
    metrics_path = save_metrics(eval_metrics, run_id)
    summary = {
        "run_id": run_id,
        "algo": algo,
        "reward": reward,
        "seed": seed,
        "maze_level": env.level,
        "maze_size": env.size,
        "difficulty": env.difficulty,
        "key_pos": list(env.key_pos),
        "door_pos": list(env.door_pos),
        "dynamic_objects": env.dynamic_objects,
        "n_dynamic_layouts": len(env._layouts),
        "success_rate": float(np.mean(eval_metrics["success"])),
        "key_pickup_rate": float(np.mean(eval_metrics["key"])),
        "door_opening_rate": float(np.mean(eval_metrics["door"])),
        "key_pickup_rate_logged": float(np.mean(key_log)) if key_log else None,
        "door_opening_rate_logged": float(np.mean(door_log)) if door_log else None,
        "metrics_path": str(metrics_path),
    }
    path = DIRS["logs"] / f"{run_id}_summary.json"
    path.write_text(json.dumps(summary, indent=2), encoding="utf-8")
    return summary

def print_summary_table(summaries):
    cols = [
        "algo", "reward", "seed", "success_rate", "key_pickup_rate",
        "door_opening_rate", "mean_steps_to_goal", "mean_steps_to_key",
        "mean_key_to_door_steps", "path_optimality",
    ]
    if not summaries:
        print("No summaries were produced.")
        return
    widths = {col: max(len(col), *(len(str(s.get(col))) for s in summaries)) for col in cols}
    print(" | ".join(col.ljust(widths[col]) for col in cols))
    print("-+-".join("-" * widths[col] for col in cols))
    for s in summaries:
        print(" | ".join(str(s.get(col)).ljust(widths[col]) for col in cols))

def run_phase2_experiment_matrix():
    summaries = []
    start_time = time.time()

    total_runs = len(ALGORITHMS_TO_RUN) * len(REWARDS_TO_RUN) * len(SEEDS_TO_RUN)
    print("runs:", total_runs)
    print("algorithms:", ALGORITHMS_TO_RUN)
    print("rewards:", REWARDS_TO_RUN)
    print("seeds:", SEEDS_TO_RUN)

    for algo in ALGORITHMS_TO_RUN:
        for reward in REWARDS_TO_RUN:
            for seed in SEEDS_TO_RUN:
                run_id = run_id_for(algo, reward, seed)
                env = DoorKeyMazeEnv(**MAZE_KWARGS, reward_type=reward)
                print("\nRUN", run_id)
                print("key=", env.key_pos, "door=", env.door_pos, "optimal=", env.optimal_path_length())

                if algo == "dqn":
                    result = train_dqn_phase2(env, cfg=DQN_RUN_CFG, seed=seed, double=False)
                    summary = save_phase2_result(run_id, env, result, algo, reward, seed)
                elif algo == "ddqn":
                    result = train_dqn_phase2(env, cfg=DQN_RUN_CFG, seed=seed, double=True)
                    summary = save_phase2_result(run_id, env, result, algo, reward, seed)
                elif algo == "ppo":
                    model, key_log, door_log = train_ppo_phase2(env, cfg=PPO_RUN_CFG, seed=seed)
                    summary = save_ppo_summary(run_id, env, model, key_log, door_log, algo, reward, seed)
                else:
                    raise ValueError(algo)

                summaries.append(summary)
                print(summary)

    print("\nelapsed seconds:", round(time.time() - start_time, 1))
    print("\nsummary table:")
    print_summary_table(summaries)
    return summaries

## Run Experiment Matrix

In [30]:
summaries = run_phase2_experiment_matrix()

runs: 1
algorithms: ['ddqn']
rewards: ['subgoal']
seeds: [0]

RUN p2_smoke_ddqn_subgoal_small_medium_seed0
key= (9, 6) door= (2, 5) optimal= 52
  [io] metrics saved -> C:\Users\pc cam dz\Desktop\RL project\results\metrics\p2_smoke_ddqn_subgoal_small_medium_seed0_metrics.npz
  [io] DQN model saved -> C:\Users\pc cam dz\Desktop\RL project\models\dqn\p2_smoke_ddqn_subgoal_small_medium_seed0.pt
{'success_rate': 0.0, 'mean_return': -60.561, 'std_return': 7.2585, 'mean_steps_to_goal': nan, 'std_steps_to_goal': nan, 'convergence_episode': None, 'sample_efficiency': None, 'path_optimality': None, 'optimal_path_len': 52, 'total_episodes': 10, 'key_pickup_rate': 0.4, 'door_opening_rate': 0.4, 'subgoal_order_rate': 0.4, 'mean_steps_to_key': 144.25, 'mean_key_to_door_steps': 116.5, 'mean_optimal_path_len': 43.6, 'run_id': 'p2_smoke_ddqn_subgoal_small_medium_seed0', 'algo': 'ddqn', 'reward': 'subgoal', 'seed': 0, 'maze_level': 'small', 'maze_size': 10, 'difficulty': 'medium', 'key_pos': [0, 9], 'do

## Inspect Saved Summaries

In [31]:
summary_files = sorted(DIRS["logs"].glob("p2_*_summary.json"))
print("Phase 2 summary files:", len(summary_files))
for path in summary_files[-5:]:
    print(path.name)

Phase 2 summary files: 1
p2_smoke_ddqn_subgoal_small_medium_seed0_summary.json


In [ ]:
SMOKE_TEST = True
RUN_LABEL = "mini"

ALGORITHMS_TO_RUN = ["dqn", "ddqn", "ppo"]
REWARDS_TO_RUN = ["sparse", "subgoal", "potential"]
SEEDS_TO_RUN = [0]

DQN_RUN_CFG = {**DQN_CFG, "episodes": 1000, "warmup": 500}
PPO_RUN_CFG = {**PPO_CFG, "total_timesteps": 100_000}

mini_summaries = run_phase2_experiment_matrix()

In [32]:
import json
from pathlib import Path

path = Path("notebooks/08_p2_data_collection.ipynb")
if not path.exists():
    path = Path("08_p2_data_collection.ipynb")

nb = json.loads(path.read_text(encoding="utf-8"))

for i, cell in enumerate(nb["cells"]):
    src = "".join(cell.get("source", []))
    if "run_phase2_experiment_matrix" in src or "mini_summaries" in src:
        print(f"\n--- CELL {i} ---")
        print(src)


--- CELL 5 ---
def run_id_for(algo, reward, seed):
    mode = globals().get("RUN_LABEL", "smoke" if SMOKE_TEST else "full")
    return f"p2_{mode}_{algo}_{reward}_{MAZE_KWARGS['level']}_{MAZE_KWARGS['difficulty']}_seed{seed}"

def tracker_optimal_path_arr(tracker):
    if hasattr(tracker, "optimal_path_arr"):
        return tracker.optimal_path_arr
    return np.full_like(tracker.steps_arr, tracker.optimal_path_len, dtype=float)

def save_phase2_result(run_id, env, result, algo, reward, seed):
    tracker = result.tracker
    metrics_path = save_metrics({
        "episodes": tracker.episodes_arr,
        "rewards": tracker.rewards,
        "steps": tracker.steps_arr,
        "success": tracker.success_arr,
        "key": tracker.key_arr,
        "door": tracker.door_arr,
        "steps_to_key": tracker.steps_to_key_arr,
        "steps_to_door": tracker.steps_to_door_arr,
        "optimal_path": tracker_optimal_path_arr(tracker),
    }, run_id)
    if algo in {"dqn", "ddqn"}:
        s